# 3. Model Training & Evaluation
## ICU XAI Research - XGBoost, TCN, and Transformer

This notebook trains and evaluates three model architectures:
- XGBoost: Gradient boosting baseline
- TCN: Temporal Convolutional Network for sequences
- Transformer: Attention-based architecture

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from models import XGBoostWrapper


In [ ]:
# Load preprocessed data
data_dir = Path.cwd().parent / 'data'
df_processed = pd.read_csv(data_dir / 'processed' / 'processed_data.csv')
feature_cols = [col for col in df_processed.columns if col not in ['RecordID', 'In-hospital_death', 'SAPS-I', 'SOFA']]

X = df_processed[feature_cols].values
y = df_processed['In-hospital_death'].astype(int).values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print('Train shape', X_train.shape)
print('Val shape', X_val.shape)
print('Test shape', X_test.shape)


In [ ]:
# Train XGBoost model
xgb_model = XGBoostWrapper(n_estimators=100, max_depth=5, learning_rate=0.1, use_label_encoder=False, eval_metric='logloss')
xgb_model.train(X_train, y_train)

xgb_pred = xgb_model.predict_proba(X_test)[:, 1]
print('AUC-ROC (XGBoost):', roc_auc_score(y_test, xgb_pred))


In [ ]:
# Train TCN model
# TODO: Replace this skeleton with a full training loop for a Temporal Convolutional Network.
print('TCN training placeholder: implement sequence batch loader, training loop, and evaluation metrics.')


In [ ]:
# Train Transformer model
# TODO: Replace this placeholder with a Transformer training routine for ICU time-series data.
print('Transformer training placeholder: implement embedding, attention, and training pipeline.')


In [ ]:
# Evaluate XGBoost model
preds = (xgb_pred >= 0.5).astype(int)
print('Confusion matrix:')
print(confusion_matrix(y_test, preds))
print('\nClassification report:')
print(classification_report(y_test, preds))

fpr, tpr, _ = roc_curve(y_test, xgb_pred)
plt.plot(fpr, tpr, label=f'XGBoost AUC={roc_auc_score(y_test, xgb_pred):.3f}')
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - XGBoost')
plt.legend()
plt.show()


In [ ]:
# Save trained model
import joblib
output_dir = Path.cwd().parent / 'outputs' / 'models'
output_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(xgb_model.model, output_dir / 'xgb_icuxai.pkl')
print(f"Saved XGBoost model to {output_dir / 'xgb_icuxai.pkl'}")
